In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
import hssm 
hssm.set_floatX("float32")

In [ ]:
# Basics
import os
import sys
import time
from matplotlib import pyplot as plt
import arviz as az  # Visualization
import pytensor  # Graph-based tensor library
import hssm
import pandas as pd
import numpy as np
import scipy as sp
from tqdm import tqdm
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# import ssms.basic_simulators # Model simulators
import hddm_wfpt
import bambi as bmb

# Setting float precision in pytensor
pytensor.config.floatX = "float32"

# from jax.config import config

import numpyro

numpyro.set_host_device_count(4)


import warnings
warnings.filterwarnings('ignore')

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        combined_df.loc[combined_df.participant==subj,
                'rpe_cat'] = pd.qcut(combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'], 3, labels=['neg', 'neutral', 'pos'])


In [ ]:


model_df = combined_df[combined_df.condition=='Day1'].dropna(subset='preward')
model_df.rename(columns={'recog_rt':'rt', 'recog_choice':'response', 'participant':'subj_idx'}, inplace=True)
model_df = model_df[['subj_idx', 'response', 'rt', 'zrpe', 'zpm', 'preward', 'phit']]
model_df['response'] = model_df['response'].apply(lambda x: 1 if x=='new' else -1)

In [ ]:
model_df

In [ ]:
ddm_model_simple = hssm.HSSM(
    model="ddm",
    data=model_df[
        ["rt", "response", "subj_idx", "zrpe", "zpm", "preward"]
    ],

    p_outlier=None,
    lapse=None,

    include=[
        # -------------------------
        # Drift rate only
        # -------------------------
        {
            "name": "v",
            "formula": "v ~ 1 + zrpe + zpm + preward + (1 | subj_idx)",
            "link": "identity",
        },

        # -------------------------
        # Other parameters: subject-level only
        # -------------------------
        {
            "name": "z",
            "formula": "z ~ 1 + (1 | subj_idx)",
            "link": "logit",
        },
        {
            "name": "a",
            "formula": "a ~ 1 + (1 | subj_idx)",
            "link": "log",
        },
        {
            "name": "t",
            "formula": "t ~ 1 + (1 | subj_idx)",
            "link": "log",
        },
    ],
)


In [ ]:
idata = ddm_model_simple.sample(
    chains = 4,
    cores = 4,
    draws = 2000,
    tune = 2000,
    idata_kwargs=dict(log_likelihood=True))